# SpaceX Falcon 9 – Web Scraping Launch Records from Wikipedia
Scrape the Falcon 9 launch tables from the Wikipedia page *List of Falcon 9 and Falcon Heavy launches* (snapshot of 9 June 2021) with `requests` + `BeautifulSoup` and convert them to a dataframe.

In [1]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

## Helper functions to process the HTML table cells

In [2]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not (column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

## Task 1: Request the Falcon 9 launch Wiki page

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {"User-Agent": "Mozilla/5.0 (IBM Data Science Capstone)"}
response = requests.get(static_url, headers=headers)
print("Status code:", response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
soup.title

Status code: 200


<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

## Task 2: Extract all column names from the HTML table header

In [4]:
html_tables = soup.find_all('table')
first_launch_table = html_tables[2]

column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## Task 3: Create a dataframe by parsing the launch HTML tables

In [5]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

def cell_text(cell):
    return cell.a.string if cell.a and cell.a.string else cell.get_text(" ", strip=True)

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)
            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(cell_text(row[2]))
            launch_dict['Payload'].append(cell_text(row[3]))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(cell_text(row[5]))
            launch_dict['Customer'].append(cell_text(row[6]))
            launch_dict['Launch outcome'].append(list(row[7].strings)[0].strip())
            launch_dict['Booster landing'].append(landing_status(row[8]).strip())

print("Rows extracted:", extracted_row)

Rows extracted: 121


In [6]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


In [7]:
df.to_csv('spacex_web_scraped.csv', index=False)
df.shape

(121, 11)